# TCC: Análise de Imagens de Termogramas Mamários com Deep Learning
### Comparativo: EfficientNet-B0 vs ResNet-50
---
Este notebook orquestra o pipeline completo de treinamento, validação e teste com aceleração por hardware na **NVIDIA RTX 5060 (CUDA 12.8)**.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from dataset import obter_dataloaders, obter_transformacoes
from modelos import criar_modelo, obter_resumo_modelo
from treinar import executar_treinamento
from avaliar_comparativo import gerar_graficos_e_relatorio_comparativo

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Visualização de Amostras Térmicas do Dataset

In [ ]:
loader_treino, loader_val, loader_teste, pesos = obter_dataloaders('splits.json', 'dataset', batch_size=8)
imagens, rotulos, pacientes = next(iter(loader_treino))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
classes_map = {0: 'Saudável (0)', 1: 'Doente (1)'}

for i, ax in enumerate(axes.flat):
    img = imagens[i].permute(1, 2, 0).cpu().numpy()
    # Desnormalizar para exibição
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(f"Paciente: {pacientes[i]}\nClasse: {classes_map[int(rotulos[i])]}", fontsize=10)
    ax.axis('off')

plt.suptitle("Amostras do Dataset de Termografia Mamária (Com Data Augmentation)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Treinamento do Modelo Principal: EfficientNet-B0

In [ ]:
metricas_teste_eff = executar_treinamento(
    nome_modelo='efficientnet_b0',
    epocas=25,
    batch_size=32,
    lr=3e-4,
    patience=8
)

## 3. Treinamento do Modelo Comparativo: ResNet-50

In [ ]:
metricas_teste_res = executar_treinamento(
    nome_modelo='resnet50',
    epocas=25,
    batch_size=32,
    lr=3e-4,
    patience=8
)

## 4. Geração dos Gráficos e Relatório Comparativo Final para o TCC

In [ ]:
gerar_graficos_e_relatorio_comparativo('.')